1. setting requirements

In [5]:
import os
import sys
import importlib.util
import ee
import folium
import geemap.foliumap as geemap
import geopandas as gpd
from IPython.display import display

c:\Users\Minutella Francesco\jk\SAR\venv\Lib\site-packages\geemap\conversion.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [6]:
ee.Authenticate()
ee.Initialize()

2. import functions as modules

In [7]:
def load_module(name, rel_path):
    module_path = os.path.abspath(rel_path)
    spec = importlib.util.spec_from_file_location(name, module_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

sys.path.append(os.path.abspath("../scripts"))

aoi_loader      = load_module("aoi_loader", "../scripts/aoi_loader.py")
download_s1     = load_module("download_s1", "../scripts/download_s1.py")
conv_db_sm     = load_module("conv_db_sm", "../scripts/conv_db_sm.py")

3. import AOI

In [8]:
print(os.path.exists("../data/aoi/aoi_1km7x16_3035.shp"))

aoi_gdf = aoi_loader.load_shapefile_as_gdf("../data/aoi/aoi_1km7x16_3035.shp")
print(type(aoi_gdf))

center = aoi_gdf.to_crs(epsg=4326).geometry.centroid.iloc[0]
print(center)
center_coords = [center.y, center.x]

m = folium.Map(location=center_coords, zoom_start=12)
folium.GeoJson(aoi_gdf).add_to(m)

m.save("map_aoi.html")
print("💾 Saved 'map_aoi.html' in the current directory")

from IPython.display import display
display(m)


True
<class 'geopandas.geodataframe.GeoDataFrame'>
POINT (12.201048660903087 44.458029809288234)
💾 Saved 'map_aoi.html' in the current directory


C:\Users\Minutella Francesco\AppData\Local\Temp\ipykernel_29216\2775533220.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center = aoi_gdf.to_crs(epsg=4326).geometry.centroid.iloc[0]


4. download from ee: SAR img and clipping it

In [9]:
""" # Earth Engine AOI
aoi_ee = aoi_loader.load_shapefile_as_ee("../data/aoi/aoi_1km7x16_3035.shp")

img = download_s1.get_s1_vv_image(aoi_ee, "2024-04-01", "2024-09-30")

Map = geemap.Map(center=[center_coords[0], center_coords[1]], zoom=12)
Map.addLayer(img, {"min": -25, "max": 0}, "SAR VV (Apr–Sep)")
Map.addLayer(ee.FeatureCollection(aoi_ee), {}, "AOI")
Map

sar_aoi = img.clip(aoi_ee)

Map = geemap.Map(center=center_coords, zoom=12)
Map.addLayer(sar_aoi, {"min": -25, "max": 0}, "SAR VV Clipped")
Map.addLayer(ee.FeatureCollection(aoi_ee), {}, "AOI")
Map
 """

' # Earth Engine AOI\naoi_ee = aoi_loader.load_shapefile_as_ee("../data/aoi/aoi_1km7x16_3035.shp")\n\nimg = download_s1.get_s1_vv_image(aoi_ee, "2024-04-01", "2024-09-30")\n\nMap = geemap.Map(center=[center_coords[0], center_coords[1]], zoom=12)\nMap.addLayer(img, {"min": -25, "max": 0}, "SAR VV (Apr–Sep)")\nMap.addLayer(ee.FeatureCollection(aoi_ee), {}, "AOI")\nMap\n\nsar_aoi = img.clip(aoi_ee)\n\nMap = geemap.Map(center=center_coords, zoom=12)\nMap.addLayer(sar_aoi, {"min": -25, "max": 0}, "SAR VV Clipped")\nMap.addLayer(ee.FeatureCollection(aoi_ee), {}, "AOI")\nMap\n '

In [10]:
aoi_ee = aoi_loader.load_shapefile_as_ee("../data/aoi/aoi_1km7x16_3035.shp")

s1_collection = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(aoi_ee)
    .filterDate("2024-06-01", "2024-07-01")
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .select('VV')
)
sar_june_mean = s1_collection.mean().clip(aoi_ee)

s2_collection = (
    ee.ImageCollection("COPERNICUS/S2_SR")
    .filterBounds(aoi_ee)
    .filterDate("2024-06-01", "2024-07-01")
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)

def add_ndvi(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return img.addBands(ndvi)

s2_ndvi = s2_collection.map(add_ndvi)
ndvi_june_mean = s2_ndvi.select('NDVI').median().clip(aoi_ee)

sar_task = ee.batch.Export.image.toDrive(
    image=sar_june_mean,
    description='sar_june_2024_mean',
    folder='earthengine',
    fileNamePrefix='sar_june_2024_mean',
    scale=10,
    fileFormat='GeoTIFF'
)
ndvi_task = ee.batch.Export.image.toDrive(
    image=ndvi_june_mean,
    description='ndvi_june_2024_median',
    folder='earthengine',
    fileNamePrefix='ndvi_june_2024_median',
    scale=10,
    fileFormat='GeoTIFF'
)

sar_task.start()
ndvi_task.start()
print("🚀 Export SAR and NDVI https://code.earthengine.google.com/tasks")


c:\Users\Minutella Francesco\jk\SAR\venv\Lib\site-packages\ee\deprecation.py:207: DeprecationWarning: 

Attention required for COPERNICUS/S2_SR! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR

  warnings.warn(warning, category=DeprecationWarning)


🚀 Export SAR and NDVI https://code.earthengine.google.com/tasks


In [11]:
import geemap
Map = geemap.Map(zoom=12)
Map.addLayer(sar_june_mean, {"min": -25, "max": 0}, "SAR VV June")
Map.addLayer(ndvi_june_mean, {"min": 0, "max": 1}, "NDVI June")
Map.addLayer(ee.FeatureCollection(aoi_ee), {}, "AOI")
Map


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [12]:
task = ee.batch.Export.image.toDrive(
    image=sar_june_mean,
    description='sar_june_2024_mean',
    folder='earthengine',
    fileNamePrefix='sar_june_2024_mean',
    scale=10,  # risoluzione tipica S1
    fileFormat='GeoTIFF'
)
task.start()
print("🚀 Export SAR giugno 2024 avviato! Controlla: https://code.earthengine.google.com/tasks")


🚀 Export SAR giugno 2024 avviato! Controlla: https://code.earthengine.google.com/tasks


5. download form ee: raster isric sand, clay, silt

In [13]:

sand_0_5 = ee.Image("projects/soilgrids-isric/sand_mean").select("sand_0-5cm_mean").divide(10) # .divide(10) = percent values
silt_0_5 = ee.Image("projects/soilgrids-isric/silt_mean").select("silt_0-5cm_mean").divide(10)
clay_0_5 = ee.Image("projects/soilgrids-isric/clay_mean").select("clay_0-5cm_mean").divide(10)

sand_clipped = sand_0_5.clip(aoi_ee)
silt_clipped = silt_0_5.clip(aoi_ee)
clay_clipped = clay_0_5.clip(aoi_ee)

soil_layers = [
    ("sand_0_5_div10", sand_clipped),
    ("silt_0_5_div10", silt_clipped),
    ("clay_0_5_div10", clay_clipped)
]

for name, image in soil_layers:
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=name,
        folder='earthengine',
        region=aoi_ee,
        scale=250,
        maxPixels=1e13
    )
    task.start()
    print(f"✅ Esportazione avviata per: {name}")


✅ Esportazione avviata per: sand_0_5_div10
✅ Esportazione avviata per: silt_0_5_div10
✅ Esportazione avviata per: clay_0_5_div10


In [14]:
""" Map = geemap.Map(center=center_coords, zoom=12)

sand_palette = ['#ffffcc','#c2e699','#78c679','#31a354','#006837']
silt_palette = ['#f7fcb9','#addd8e','#31a354']
clay_palette = ["#ecc052","#c68120","#A34C00","#6b2608"]

Map.addLayer(sand_clipped, {"min": 0, "max": 100, "palette": sand_palette}, "Sand 0–5cm")
Map.addLayer(silt_clipped, {"min": 0, "max": 100, "palette": silt_palette}, "Silt 0–5cm")
Map.addLayer(clay_clipped, {"min": 0, "max": 100, "palette": clay_palette}, "Clay 0–5cm")

Map.addLayer(ee.FeatureCollection(aoi_ee), {}, "AOI")

Map
 """

' Map = geemap.Map(center=center_coords, zoom=12)\n\nsand_palette = [\'#ffffcc\',\'#c2e699\',\'#78c679\',\'#31a354\',\'#006837\']\nsilt_palette = [\'#f7fcb9\',\'#addd8e\',\'#31a354\']\nclay_palette = ["#ecc052","#c68120","#A34C00","#6b2608"]\n\nMap.addLayer(sand_clipped, {"min": 0, "max": 100, "palette": sand_palette}, "Sand 0–5cm")\nMap.addLayer(silt_clipped, {"min": 0, "max": 100, "palette": silt_palette}, "Silt 0–5cm")\nMap.addLayer(clay_clipped, {"min": 0, "max": 100, "palette": clay_palette}, "Clay 0–5cm")\n\nMap.addLayer(ee.FeatureCollection(aoi_ee), {}, "AOI")\n\nMap\n '

6. stack isric: merged sand, clay, silt

In [15]:
soil_stack = sand_clipped.rename("sand").addBands(
    silt_clipped.rename("silt")).addBands(
    clay_clipped.rename("clay"))

sampled = soil_stack.sampleRegions(
    collection=soil_stack,
    scale=250,
    geometries=True
)

task = ee.batch.Export.image.toDrive(
    image=soil_stack,
    description='soil_stack_raster',
    folder='earthengine',
    fileNamePrefix='soil_stack',
    scale=250,
    region=aoi_ee,
    fileFormat='GeoTIFF'
)
task.start()
print("🚀 Export raster!")
print("🚀 Export avviato! Controlla: https://code.earthengine.google.com/tasks")


🚀 Export raster!
🚀 Export avviato! Controlla: https://code.earthengine.google.com/tasks


7. centroids on stack raster sand, clay, silt

In [ ]:
soil_stack = sand_clipped.rename("sand").addBands(
    silt_clipped.rename("silt")).addBands(
    clay_clipped.rename("clay"))

rpoints_ee = soil_stack.sample(
    region=aoi_ee,
    scale=250,
    geometries=True
)


In [ ]:
task = ee.batch.Export.table.toDrive(
    collection=rpoints_ee,
    description='rpoints_soil',
    folder='earthengine',
    fileFormat='SHP'
)
task.start()
print("🚀 Export avviato! Controlla: https://code.earthengine.google.com/tasks")

🚀 Export avviato! Controlla: https://code.earthengine.google.com/tasks


In [ ]:
rpoints_soil = aoi_loader.load_shapefile_as_gdf("../output/isric/points/rpoints_soil.shp")

def texture_class_usda(sand, clay, silt):
    if sand is None or silt is None or clay is None:
        return 'n'
    if sand >= 0 and sand <= 45 and clay >= 40 and clay <= 100 and silt >= 0 and silt <= 40:
        return 'Cl'
    elif sand <= 65 and sand >= 45 and clay >= 35 and clay <= 55 and silt >= 0 and silt <= 20:
        return 'SaCl'
    elif sand >= 0 and sand <= 20 and clay >= 40 and clay <= 60 and silt >= 40 and silt <= 60:
        return 'SiCl'
    elif sand >= 20 and sand <= 45 and clay >= 25 and clay <= 40 and silt >= 15 and silt <= 55:
        return 'ClLo'
    elif sand >= 0 and sand <= 20 and clay >= 25 and clay <= 40 and silt >= 40 and silt <= 75:
        return 'SiClLo'
    elif sand >= 45 and sand <= 80 and clay >= 20 and clay <= 35 and silt >= 0 and silt <= 25:
        return 'SaClLo'
    elif sand >= 25 and sand <= 55 and clay >= 5 and clay <= 25 and silt >= 25 and silt <= 50:
        return 'Lo'
    elif sand >= 85 and sand <= 100 and clay >= 0 and clay <= 10 and silt >= 0 and silt <= 15:
        return 'Sa'
    elif sand >= 70 and sand <= 90 and clay >= 0 and clay <= 15 and silt >= 0 and silt <= 30:
        return 'LoSa'
    elif sand >= 45 and sand <= 85 and clay >= 0 and clay <= 20 and silt >= 0 and silt <= 50:
        return 'SaLo'
    elif sand >= 0 and sand <= 50 and clay >= 0 and clay <= 25 and silt >= 50 and silt <= 85:
        return 'SiLo'
    elif sand >= 0 and sand <= 20 and clay >= 0 and clay <= 15 and silt >= 80 and silt <= 100:
        return 'y'
    else:
        return 'n'

rpoints_soil['texture_class'] = rpoints_soil.apply(
    lambda row: texture_class_usda(row['sand'], row['clay'], row['silt']), axis=1
)



In [ ]:
rpoints_soil.to_file("../output/isric/points/rpoints_soil.shp")


C:\Users\Minutella Francesco\AppData\Local\Temp\ipykernel_27976\1381772949.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  rpoints_soil.to_file("../output/isric/points/rpoints_soil.shp")
c:\Users\Minutella Francesco\jk\SAR\venv\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'texture_class' to 'texture__1'
  ogr_write(


In [20]:
rpoints_soil = gpd.read_file("../output/isric/points/rpoints_soil.shp")
rpoints_soil.head()

,sand,silt,clay,texture_cl,texture__1,geometry
0,13.5,47.2,39.3,SiClLo,SiClLo,POINT (12.24778 44.5965)
1,13.6,47.0,39.4,SiClLo,SiClLo,POINT (12.25086 44.5965)
2,13.5,49.0,37.5,SiClLo,SiClLo,POINT (12.25394 44.5965)
3,12.2,50.5,37.3,SiClLo,SiClLo,POINT (12.25702 44.5965)
4,12.3,49.6,38.1,SiClLo,SiClLo,POINT (12.2601 44.5965)


In [ ]:
import geemap

rpoints_soil_ee = geemap.geopandas_to_ee(rpoints_soil)


In [ ]:
import geopandas as gpd
import rasterio

gdf = gpd.read_file("../output/isric/points/rpoints_soil.shp")

rasters = {
    "VV": "../output/s1/sar_june_2024_mean.tif",
    "NDVI": "../output/s1/ndvi_june_2024_mean.tif",
    "SM_Copernicus": "../output/s1/SM1km_10giu24.tif",
}

with rasterio.open(list(rasters.values())[0]) as ref_raster:
    gdf = gdf.to_crs(ref_raster.crs)

coords = [(geom.x, geom.y) for geom in gdf.geometry]

for colname, rasterfile in rasters.items():
    with rasterio.open(rasterfile) as src:
        vals = [x[0] for x in src.sample(coords)]
        gdf[colname] = vals

gdf = gdf.dropna(subset=["VV", "NDVI", "SM_Copernicus", "sand", "silt", "clay"])

gdf.to_file("../output/isric/points/rpoints_soil_with_vars.shp")
gdf.to_csv("../output/isric/points/rpoints_soil_with_vars.csv", index=False)

gdf.head()


,sand,silt,clay,texture_cl,texture__1,geometry,VV,NDVI,SM_Copernicus
0,13.5,47.2,39.3,SiClLo,SiClLo,POINT (12.24778 44.5965),-10.886836,0.507595,56
1,13.6,47.0,39.4,SiClLo,SiClLo,POINT (12.25086 44.5965),-11.550593,0.520076,61
2,13.5,49.0,37.5,SiClLo,SiClLo,POINT (12.25394 44.5965),-12.751440,0.426539,61
3,12.2,50.5,37.3,SiClLo,SiClLo,POINT (12.25702 44.5965),-15.975950,0.203824,61
4,12.3,49.6,38.1,SiClLo,SiClLo,POINT (12.2601 44.5965),-9.425989,0.556047,84
